# IPL Sentiment Pipeline Selector Lab

This completed notebook compares 4 vectorizers and 4 classifiers for the IPL sentiment challenge, then exports the selected recipes and results.

## Step 0: Install/import libraries

If needed, run `pip install -r requirements.txt` before opening this notebook.

In [ ]:
import json, time, tempfile
from pathlib import Path

import joblib
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
print('Libraries loaded successfully.')

## Step 1: Load the 80-row dataset

In [ ]:
DATA_URL_80R = 'https://raw.githubusercontent.com/HimanshuKhale/ipl_sentiment/main/ipl_sentiment_small_dataset.csv'
LOCAL_DATA_PATH_80R = Path('dataset/ipl_sentiment_small_dataset.csv')

try:
    df_80 = pd.read_csv(DATA_URL_80R)
    print('Loaded dataset from GitHub')
except Exception:
    df_80 = pd.read_csv(LOCAL_DATA_PATH_80R)
    print('Loaded local dataset')

print('Shape:', df_80.shape)
df_80.head()

## Step 2: Clean and standardize the dataset

In [ ]:
required_columns = ['statement', 'subject_keyword', 'sentiment']
missing = [col for col in required_columns if col not in df_80.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df_80 = df_80.dropna(subset=required_columns).copy()
df_80['statement'] = df_80['statement'].astype(str)
df_80['subject_keyword'] = df_80['subject_keyword'].astype(str)
df_80['sentiment'] = df_80['sentiment'].astype(str).str.lower().str.strip()
df_80['model_text'] = df_80['subject_keyword'] + ' [SEP] ' + df_80['statement']

print(df_80['sentiment'].value_counts())
df_80[['model_text', 'sentiment']].head()

## Step 3: Train/test split

In [ ]:
X = df_80['model_text']
y = df_80['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))

## Step 4: Selected vectorizer and classifier recipes

In [ ]:
vectorizer_recipes = [
    {'id': 'count_basic', 'name': 'CountVectorizer Basic', 'family': 'count', 'type': 'CountVectorizer', 'params': {'lowercase': True, 'max_features': 3000}, 'complexity': 'low'},
    {'id': 'tfidf_word_1_2', 'name': 'TF-IDF Word 1-2 grams', 'family': 'tfidf_word', 'type': 'TfidfVectorizer', 'params': {'lowercase': True, 'ngram_range': (1, 2), 'max_features': 4000}, 'complexity': 'low'},
    {'id': 'tfidf_char_3_5', 'name': 'TF-IDF Character 3-5 grams', 'family': 'tfidf_char', 'type': 'TfidfVectorizer', 'params': {'analyzer': 'char', 'ngram_range': (3, 5), 'max_features': 5000}, 'complexity': 'medium'},
    {'id': 'hashing_word', 'name': 'HashingVectorizer Word', 'family': 'hashing', 'type': 'HashingVectorizer', 'params': {'alternate_sign': False, 'n_features': 2048, 'lowercase': True}, 'complexity': 'medium'},
]

classifier_recipes = [
    {'id': 'multinomial_nb', 'name': 'Multinomial Naive Bayes', 'family': 'probabilistic', 'type': 'MultinomialNB', 'params': {}, 'complexity': 'low'},
    {'id': 'logreg_balanced', 'name': 'Logistic Regression Balanced', 'family': 'linear', 'type': 'LogisticRegression', 'params': {'max_iter': 1000, 'class_weight': 'balanced', 'random_state': RANDOM_STATE}, 'complexity': 'low'},
    {'id': 'linear_svc', 'name': 'LinearSVC', 'family': 'margin_based', 'type': 'LinearSVC', 'params': {'random_state': RANDOM_STATE}, 'complexity': 'medium'},
    {'id': 'sgd_log', 'name': 'SGDClassifier Logistic Loss', 'family': 'linear', 'type': 'SGDClassifier', 'params': {'loss': 'log_loss', 'max_iter': 1000, 'random_state': RANDOM_STATE}, 'complexity': 'medium'},
]

pd.DataFrame(vectorizer_recipes), pd.DataFrame(classifier_recipes)

## Step 5: Validate diversity and train all 16 combinations

In [ ]:
def validate_recipe_diversity(vectorizers, classifiers):
    v_families = sorted(set(v['family'] for v in vectorizers))
    c_families = sorted(set(c['family'] for c in classifiers))
    heavy = [c for c in classifiers if c.get('complexity') == 'high']
    print('Vectorizer count:', len(vectorizers))
    print('Classifier count:', len(classifiers))
    print('Vectorizer families:', v_families)
    print('Classifier families:', c_families)
    print('Heavy classifiers:', [c['id'] for c in heavy])
    assert len(vectorizers) >= 4 and len(classifiers) >= 4
    assert len(v_families) >= 3 and len(c_families) >= 3
    assert len(heavy) <= 1
    print('Diversity check passed.')

validate_recipe_diversity(vectorizer_recipes, classifier_recipes)

def make_vectorizer(recipe):
    cls = {'CountVectorizer': CountVectorizer, 'TfidfVectorizer': TfidfVectorizer, 'HashingVectorizer': HashingVectorizer}[recipe['type']]
    return cls(**recipe['params'])

def make_classifier(recipe):
    params = dict(recipe['params'])
    if recipe['type'] in {'LogisticRegression', 'LinearSVC', 'SGDClassifier'}:
        params['random_state'] = RANDOM_STATE
    cls = {'MultinomialNB': MultinomialNB, 'LogisticRegression': LogisticRegression, 'LinearSVC': LinearSVC, 'SGDClassifier': SGDClassifier}[recipe['type']]
    return cls(**params)

def model_size_kb(model):
    with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as f:
        path = Path(f.name)
    joblib.dump(model, path)
    size = path.stat().st_size / 1024
    path.unlink()
    return size

def complexity_penalty(v_recipe, c_recipe):
    levels = {'low': 0, 'medium': 1, 'high': 2}
    return levels.get(v_recipe.get('complexity', 'low'), 0) + levels.get(c_recipe.get('complexity', 'low'), 0)

results = []
models = {}

for v in vectorizer_recipes:
    for c in classifier_recipes:
        name = f"{v['id']}__{c['id']}"
        pipe = Pipeline([('vectorizer', make_vectorizer(v)), ('classifier', make_classifier(c))])
        start_train = time.perf_counter()
        pipe.fit(X_train, y_train)
        train_time = time.perf_counter() - start_train
        start_pred = time.perf_counter()
        pred = pipe.predict(X_test)
        pred_time = time.perf_counter() - start_pred
        acc = accuracy_score(y_test, pred)
        macro_f1 = f1_score(y_test, pred, average='macro', zero_division=0)
        weighted_f1 = f1_score(y_test, pred, average='weighted', zero_division=0)
        precision_macro = precision_score(y_test, pred, average='macro', zero_division=0)
        recall_macro = recall_score(y_test, pred, average='macro', zero_division=0)
        size_kb = model_size_kb(pipe)
        performance_score = (macro_f1 * 60) + (acc * 25) + (weighted_f1 * 15)
        speed_score = max(0, 100 - (train_time * 10) - (pred_time * 100))
        size_score = max(0, 100 - (size_kb / 20))
        efficiency_score = (speed_score * 0.6) + (size_score * 0.4)
        suitability_score = max(0, 100 - complexity_penalty(v, c) * 15)
        final_score = performance_score * 0.55 + efficiency_score * 0.20 + suitability_score * 0.15 + 100 * 0.10
        models[name] = pipe
        results.append({
            'pipeline_id': name,
            'vectorizer_id': v['id'], 'vectorizer_family': v['family'], 'vectorizer_complexity': v['complexity'],
            'classifier_id': c['id'], 'classifier_family': c['family'], 'classifier_complexity': c['complexity'],
            'status': 'completed', 'accuracy': acc, 'macro_f1': macro_f1, 'weighted_f1': weighted_f1,
            'precision_macro': precision_macro, 'recall_macro': recall_macro,
            'train_time_sec': train_time, 'prediction_time_sec': pred_time, 'model_size_kb': size_kb,
            'performance_score': performance_score, 'efficiency_score': efficiency_score,
            'suitability_score': suitability_score, 'final_score': final_score, 'error': '',
        })

results_df = pd.DataFrame(results).sort_values('final_score', ascending=False).reset_index(drop=True)
results_df.insert(0, 'rank', range(1, len(results_df) + 1))
results_df

## Step 6: Save results, model, and submission manifest

In [ ]:
best_name = results_df.loc[0, 'pipeline_id']
best_model = models[best_name]

results_df.to_csv('results_80R.csv', index=False)
joblib.dump(best_model, 'best_ipl_sentiment_pipeline.pkl')

submission = {
    'dataset': 'ipl_sentiment_small_dataset.csv',
    'selected_vectorizers': vectorizer_recipes,
    'selected_classifiers': classifier_recipes,
    'top_pipeline': results_df.iloc[0].to_dict(),
    'hypothesis': 'Lightweight, diverse vectorizers and linear/Naive Bayes classifiers should generalize better than bulky duplicate models on a small IPL sentiment dataset.'
}

with open('submission_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(submission, f, indent=2, default=str)

print('Best pipeline:', best_name)
print(classification_report(y_test, best_model.predict(X_test)))